# Alternative Data

## Problem Definition

**Question.** Does the local AAPL 2025 news Parquet contain timestamped observed articles suitable for event formation?

**Role in the workflow.** Establish the immutable news source without calling an external service.

**Inputs.** `data/research_data/alternative/data/aapl_2025-01-01_2025-12-31.parquet`.

**Outputs.** A validated news table and a publication-time quality summary; the source file is unchanged.

**Why this method.** Publication timestamps allow articles to be aligned only to bars completed after the news became observable.

**Assumptions.** `created_at` is the first observable article time; article text may still contain vendor-specific omissions or duplicates.

**Handoff.** The local news path to `alternative_sentiment_scores.ipynb` and `event_labeling.ipynb`.


## Real Data Check

No target or future market outcome is joined here. The eventual holdout boundary remains unknown until event construction, preventing exploratory use of its labels.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
news_path = PROJECT_ROOT / "data/research_data/alternative/data/aapl_2025-01-01_2025-12-31.parquet"

news = pd.read_parquet(news_path)
required_columns = {
    "id", "headline", "source", "url", "summary", "created_at",
    "updated_at", "symbols", "author", "content",
}
assert required_columns.issubset(news.columns)

news["created_at"] = pd.to_datetime(news["created_at"], utc=True)
news["updated_at"] = pd.to_datetime(news["updated_at"], utc=True)
news = news.sort_values("created_at", ignore_index=True)

assert news["created_at"].dt.year.eq(2025).all()
assert news["symbols"].astype("string").str.contains("AAPL", na=False).all()

quality = pd.Series(
    {
        "rows": len(news),
        "start_utc": news["created_at"].min(),
        "end_utc": news["created_at"].max(),
        "duplicate_id": int(news["id"].duplicated().sum()),
        "duplicate_time_headline": int(news.duplicated(["created_at", "headline"]).sum()),
        "missing_headline": int(news["headline"].isna().sum()),
        "missing_content": int(news["content"].isna().sum()),
    },
    name="value",
)
display(quality.to_frame())
display(news[["created_at", "headline", "source"]].head())


## Results, Limitations, and Handoff

News coverage is vendor-dependent and may not represent every information event. Repeated time-headline pairs are reported and are deduplicated only when events are formed.

The next notebook receives the validated local news Parquet path. No conclusion in this notebook is evidence of live-trading profitability.
